## Predict Future Sales

This project predicts next month's sales count for each shop-item pair, for a Russian software retailer (1C Company). It is the final project of the "How to Win a Data Science Competition" Coursera course, and a good introduction to time series feature engineering.

## Approach
1. Load the data and aggregate daily transactions into monthly shop-item sales
2. Create calendar features and clip the target to [0, 20]
3. Validate by time: train on earlier months and validate on the most recent month
4. Train a baseline gradient boosting model
5. Generate predictions and create the submission file
6. Submit to Kaggle and record the score

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c competitive-data-science-predict-future-sales
!unzip -oq competitive-data-science-predict-future-sales.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.2 MB/s eta 0:00:00
100% 15.1M/15.1M [00:00<00:00, 93.4MB/s]



In [2]:
import pandas as pd
import numpy as np

sales = pd.read_csv("sales_train.csv")
test = pd.read_csv("test.csv")
items = pd.read_csv("items.csv")

In [3]:
sales.shape, test.shape, items.shape

((2935849, 6), (214200, 3), (22170, 3))

In [4]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2935849 entries, 0 to 2935848
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   date            object 
 1   date_block_num  int64  
 2   shop_id         int64  
 3   item_id         int64  
 4   item_price      float64
 5   item_cnt_day    float64
dtypes: float64(2), int64(3), object(1)
memory usage: 134.4+ MB


In [5]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214200 entries, 0 to 214199
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   ID       214200 non-null  int64
 1   shop_id  214200 non-null  int64
 2   item_id  214200 non-null  int64
dtypes: int64(3)
memory usage: 4.9 MB


In [7]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22170 entries, 0 to 22169
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   item_name         22170 non-null  object
 1   item_id           22170 non-null  int64 
 2   item_category_id  22170 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 519.7+ KB


In [8]:
sales.head(3)

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.0,1.0
1,03.01.2013,0,25,2552,899.0,1.0
2,05.01.2013,0,25,2552,899.0,-1.0


In [9]:
items.sample(1)

,item_name,item_id,item_category_id
17825,РЕАЛЬНЫЕ ПАЦАНЫ 2 СЕЗОН 2 ЧАСТЬ,17825,40


In [10]:
sales.sample(1)

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
1761131,12.06.2014,17,59,11222,399.0,1.0


In [11]:
test.head()

,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233
3,3,5,5232
4,4,5,5268


In [12]:
monthly = sales.groupby(["date_block_num", "shop_id", "item_id"])["item_cnt_day"].sum().reset_index()
monthly = monthly.rename(columns={"item_cnt_day": "item_cnt_month"})
monthly["item_cnt_month"] = monthly["item_cnt_month"].clip(0, 20)

In [13]:
print(monthly.shape)

(1609124, 4)


In [14]:
monthly["item_cnt_month"].describe()

,item_cnt_month
count,1.609124e+06
mean,2.022806e+00
std,2.577964e+00
min,0.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,2.000000e+00
max,2.000000e+01


In [15]:
monthly["month"] = monthly["date_block_num"] % 12 + 1
monthly["year"] = 2013 + monthly["date_block_num"] // 12

In [16]:
test["date_block_num"] = 34
test["month"] = test["date_block_num"] % 12 + 1
test["year"] = 2013 + test["date_block_num"] // 12

In [17]:
print(monthly[["date_block_num", "month", "year"]].tail(3))

         date_block_num  month  year
1609121              33     10  2015
1609122              33     10  2015
1609123              33     10  2015


In [18]:
print(test[["date_block_num", "month", "year"]].head(3))

   date_block_num  month  year
0              34     11  2015
1              34     11  2015
2              34     11  2015


In [19]:
train = monthly[monthly["date_block_num"] < 33]
val = monthly[monthly["date_block_num"] == 33]

In [20]:
x_train = train.drop(columns=["item_cnt_month"])
y_train = train["item_cnt_month"]
x_val = val.drop(columns=["item_cnt_month"])
y_val = val["item_cnt_month"]

In [21]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

model = HistGradientBoostingRegressor(max_iter=300, random_state=42)
model.fit(x_train, y_train)

pred = np.clip(model.predict(x_val), 0, 20)
rmse = mean_squared_error(y_val, pred) ** 0.5
print("Validation RMSE:", round(rmse, 4))

Validation RMSE: 2.374


In [22]:
lag = monthly[["date_block_num", "shop_id", "item_id", "item_cnt_month"]].copy()
lag["date_block_num"] = lag["date_block_num"] + 1
lag = lag.rename(columns={"item_cnt_month": "prev_month_cnt"})

In [23]:
monthly = monthly.merge(lag, on=["date_block_num", "shop_id", "item_id"], how="left")
monthly["prev_month_cnt"] = monthly["prev_month_cnt"].fillna(0)

In [24]:
test = test.merge(lag, on=["date_block_num", "shop_id", "item_id"], how="left")
test["prev_month_cnt"] = test["prev_month_cnt"].fillna(0)

In [25]:
monthly["prev_month_cnt"].describe()

,prev_month_cnt
count,1.609124e+06
mean,1.232208e+00
std,2.707183e+00
min,0.000000e+00
25%,0.000000e+00
50%,0.000000e+00
75%,1.000000e+00
max,2.000000e+01


In [29]:
train = monthly[monthly["date_block_num"] < 33]
val = monthly[monthly["date_block_num"] == 33]

In [32]:
x_train = train.drop(columns=["item_cnt_month"])
y_train = train["item_cnt_month"]
x_val = val.drop(columns=["item_cnt_month"])
y_val = val["item_cnt_month"]

In [33]:
model = HistGradientBoostingRegressor(max_iter=300, random_state=42)
model.fit(x_train, y_train)

pred = np.clip(model.predict(x_val), 0, 20)
rmse = mean_squared_error(y_val, pred) ** 0.5
print("Validation RMSE:", round(rmse, 4))

Validation RMSE: 2.1774


In [37]:
import numpy as np
x = monthly.drop(columns=["item_cnt_month"])
y = monthly["item_cnt_month"]

model.fit(x, y)

pred_test = np.clip(model.predict(test[x.columns]), 0, 20)

pd.DataFrame({"ID": test["ID"], "item_cnt_month": pred_test}).to_csv("submission.csv", index=False)

In [38]:
!kaggle competitions submit -c competitive-data-science-predict-future-sales -f submission.csv -m "HistGradientBoosting with lag feature"

100% 5.14M/5.14M [00:00<00:00, 18.8MB/s]
99 submissions remaining today.
Successfully submitted to Predict Future Sales

In [39]:
import pickle, os
pickle.dump(model, open("futuresales_model.pkl", "wb"))
print("size MB:", round(os.path.getsize("futuresales_model.pkl") / 1e6, 2))

size MB: 1.07
